# Disaster Impact Assessment (DIA) Challenge
### Multimodal Deep Learning for Building Damage Classification
#### VIA Madrid Summer School in Vision and Artificial Intelligence, 2nd Edition - 15th-17th June 2026

#### Author: Miguel Ángel Fernández-Torres
#### Code implementation assisted by [Claude](https://claude.ai) (Anthropic).

---

This notebook guides you through every stage of the **Disaster Impact Assessment (DIA) challenge**:
from loading multimodal data, through training a baseline classifier,
to interpreting the predictions with explainability tools.

Disaster response requires a rapid and accurate understanding of affected infrastructure to optimize humanitarian aid and rescue operations. Traditionally, building damage assessments are performed manually or on-site, which is dangerous, time-consuming, and delays critical assistance. The goal of this challenge is to develop a **multimodal deep learning model** that automatically classifies the level of structural damage sustained by buildings after a natural disaster, combining satellite imagery with ERA-5 climate data.

---

### Dataset: xBD

The dataset used in this challenge is **xBD** ([Gupta et al., 2019](https://arxiv.org/abs/1911.09296)), one of the largest building damage assessment benchmarks to date. It contains high-resolution satellite imagery (< 0.8 m GSD) covering **19 different natural disasters** worldwide — including earthquakes, floods, wildfires, hurricanes, and tsunamis — with over **850,000 annotated building polygons** spanning more than **45,000 km²**.

Each disaster is represented by pairs of **pre-event** and **post-event** images of size 1024 × 1024 pixels. From these, individual building patches of **64 × 64 pixels** are extracted using the polygon annotations, one patch per building. These patches are the unit of classification.

| Split | Images (1024²) | Building patches |
|-------|---------------|-----------------|
| Train | 18,336 | 61,171 |
| Validation | 1,866 | 8,495 |
| Test | 1,866 | 22,222 |

For more information and access to the full competition, visit [xView2](https://www.xview2.org/).

---

### Task

Classify each building patch into one of four ordinal damage categories:

| Class | Label | Visual Indicators |
|-------|-------|------------------|
| 0 | **no-damage** | Structure fully intact; no visible roof damage, burns, or debris |
| 1 | **minor-damage** | Partially burnt, surrounded by water, missing roof elements, or hairline cracks |
| 2 | **major-damage** | Partial wall or roof collapse; structure seriously compromised by environmental factors (e.g. volcanic flow, flooding) |
| 3 | **destroyed** | Charred remains, complete collapse, or building no longer present after the disaster |

---

### Multimodal Inputs

Each training sample consists of three aligned data sources:

- A **pre-event** RGB satellite patch (64 × 64 pixels) — the building before the disaster
- A **post-event** RGB satellite patch (64 × 64 pixels) — the same building after the disaster
- A **climate time series**: 10 ERA-5 environmental variables recorded on a 20 × 20 spatial grid over the full period surrounding the disaster event

The ERA-5 variables capture atmospheric and surface conditions (wind speed, precipitation, temperature, pressure, etc.) that are directly linked to the type and severity of the disaster. Exploiting this temporal climate context, alongside the visual change between pre and post images, is what makes this a genuinely **multimodal** challenge.

**Output**: one damage class label (0 – 3) per building patch.

---

### Baseline System

This notebook introduces a **simplified baseline model** that serves as your starting point:

- Processes pre and post images through a shared CNN backbone
- Encodes the climate time series with a lightweight temporal encoder
- Uses a **self-attention mechanism** to identify the most relevant climate timesteps, *without* requiring explicit event-timing labels
- Fuses all modalities and predicts the damage class

The baseline intentionally leaves significant room for improvement. Sections 3–5 walk you through training and evaluation.

---

### How to Use This Notebook
1. Run every cell in order at least once to reproduce the baseline results.
2. Cells labelled 🔧 are your entry points for modifications.
3. **Part 6** lists research directions organiszeed by topic. Pick your favourites or others related.

> ⚠️ Do **not** modify `patch_size` (fixed at 64 pixels) or the test-set evaluation pipeline.


## Environment Setup

In [ ]:
!pip install opencv-python wandb lightning captum

In [ ]:
# ── 0.1  System-level fixes for headless rendering ──────────────────────────
import os, sys
os.environ['QT_QPA_PLATFORM'] = 'offscreen'   # prevent Qt/OpenCV crash
import matplotlib
matplotlib.use('Agg')                          # non-interactive backend

# ── 0.2  Project root ─────────────────────────────────────────────────────── 
PROJECT_DIR = '' #'/path/to/VIA-DIA'               # ← change to your clone path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# ── 0.3  Core imports ────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# ── 0.4  Check preprocessed cache ───────────────────────────────────────────
# Run preprocess_xbd.py ONCE before this notebook if the cache is missing.
CACHE_DIR = '/mnt/database/xBD' # os.path.join(PROJECT_DIR, 'databases/xBD')

for split in ('cache_train', 'cache_val', 'cache_test'):
    path = os.path.join(CACHE_DIR, split)
    status = '✓' if os.path.isdir(path) else '✗  MISSING'
    print(f'  {status}  {path}')


## 1) The Dataset

The `xBDClimateDataset` wraps the preprocessed cache.  Each call to
`__getitem__` returns a **9-tuple**:

```
patches_pre    (3, 64, 64)         float32  – pre-disaster RGB patch
patches_post   (3, 64, 64)         float32  – post-disaster RGB patch
mask           (64, 64)            float32  – building mask
climate        (10, T, 20, 20)     float32  – ERA-5 time series
event_labels   (T,)                float32  – event-label annotations
label_pre      ()                  int64    – pre-event damage label
label_post     ()                  int64    – post-event damage label  ← target
img_path       str                          – path to source image
patch_idx      ()                  int64    – global sample index
```

`T` is the number of climate timesteps (up to ~329 for daily ERA-5).


In [ ]:
from databases.xBDClimate_database import xBDClimateDataset, get_dataloaders

# Load train split (augmentation disabled here for clean visualisation)
train_dataset_vis = xBDClimateDataset(
    os.path.join(CACHE_DIR, 'cache_train'), augment=False
)

print(f'Training samples : {len(train_dataset_vis):,}')
print(f'Climate variables: {train_dataset_vis.climate_variable_names}')
print(f'Damage classes   : {train_dataset_vis.damage_classes}')

# Inspect one sample
sample = train_dataset_vis[0]
names  = ['pre-image','post-image','mask','climate','event_labels',
          'label_pre','label_post','img_path','patch_idx']
for n, s in zip(names, sample):
    if hasattr(s, 'shape'):
        print(f'  {n:<15}: {tuple(s.shape)}  dtype={s.dtype}')
    else:
        print(f'  {n:<15}: {s}')


### 1.1)  Class Distribution

In [ ]:
from collections import Counter

labels = train_dataset_vis.label_post          # flat array over all patches
counts = Counter(int(l) for l in labels)
names  = list(train_dataset_vis.damage_classes.keys())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, [counts.get(i,0) for i in range(4)],
              color=['#2196F3','#4CAF50','#FF9800','#F44336'])
ax.bar_label(bars, fmt='%d')
ax.set_title('Class distribution – training set')
ax.set_ylabel('Number of patches')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print({n: counts.get(i,0) for i, n in enumerate(names)})


### 1.2)  Visualize Sample Patches

In [ ]:
def denorm(patch, mean, std):
    """Reverse normalization for display. Returns uint8 HWC array."""
    img = patch.numpy() * 3 * std.reshape(3,1,1) + mean.reshape(3,1,1)
    img = np.clip(img, 0, 255).astype(np.uint8).transpose(1,2,0)
    return img

mean = train_dataset_vis.patch_mean
std  = train_dataset_vis.patch_std

# Layout: 4 rows (one per damage class) x 4 cols (pre1, post1, pre2, post2)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
shown = {0: [], 1: [], 2: [], 3: []}

for idx in range(len(train_dataset_vis)):
    s   = train_dataset_vis[idx]
    lbl = int(s[6])
    if len(shown[lbl]) >= 2:
        continue
    shown[lbl].append(s)
    if all(len(v) == 2 for v in shown.values()):
        break

for lbl in range(4):
    for ex in range(2):           # two examples per class
        s   = shown[lbl][ex]
        col_pre  = ex * 2          # 0 or 2
        col_post = ex * 2 + 1     # 1 or 3
        axes[lbl, col_pre ].imshow(denorm(s[0], mean, std))
        axes[lbl, col_pre ].set_title(f"{names[lbl]}pre #{ex+1}", fontsize=8)
        axes[lbl, col_post].imshow(denorm(s[1], mean, std))
        axes[lbl, col_post].set_title(f"{names[lbl]}post #{ex+1}", fontsize=8)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Two examples per damage class (pre / post)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_patches.png', dpi=150)
plt.show()


### 1.3)  Visualize a Climate Time Series

In [ ]:
# ── ERA-5 variable metadata ──────────────────────────────────────────────────
# 10 ERA-5 variables stored in the climate tensor (one channel per variable)
ERA5_META = {
    '2m_temperature'                 : ('T2m',   '°C (norm.)',  '#e53935'),
    'total_precipitation'            : ('TP',    'mm (norm.)',  '#1e88e5'),
    '10m_u_component_of_wind'        : ('U10',   'm/s (norm.)', '#43a047'),
    '10m_v_component_of_wind'        : ('V10',   'm/s (norm.)', '#8e24aa'),
    'volumetric_soil_water_layer_1'  : ('VSW1',  'm³/m³ (norm.)','#6d4c41'),
    'surface_solar_radiation_downwards':('SSRD', 'J/m² (norm.)','#fb8c00'),
    'surface_latent_heat_flux'       : ('SLHF',  'J/m² (norm.)','#00acc1'),
    'leaf_area_index_high_vegetation': ('LAI-HV','m²/m² (norm.)','#7cb342'),
    'surface_pressure'               : ('SP',    'Pa (norm.)',  '#5e35b1'),
    'potential_evaporation'          : ('PEV',   'm (norm.)',   '#f4511e'),
}

# Event label definitions (from xBDClimate_database.py)
EVENT_LABEL_NAMES  = {
    0: 'no event',
    1: 'disaster period',
    2: 'pre-image date',
    3: 'post-image date',
    4: 'climate series end',
}
EVENT_LABEL_COLORS = {
    1: '#1565C0',   # blue   – disaster period
    2: '#2E7D32',   # green  – pre-image date
    3: '#C62828',   # red    – post-image date
    4: '#6A1B9A',   # purple – series end
}

# ── Load one sample ──────────────────────────────────────────────────────────
s      = train_dataset_vis[42]
clim   = s[3].numpy()   # (10, T, 20, 20)
evlbl  = s[4].numpy()   # (T,)  integer labels 0-4
T      = clim.shape[1]

# Spatially average to get one curve per variable: (10, T)
clim_mean = clim.mean(axis=(2, 3))

varnames = train_dataset_vis.climate_variable_names   # list of 10 strings
meta_vals = [ERA5_META[n] for n in varnames]          # (abbrev, unit, colour)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 2, figsize=(16, 14), sharex=True)

for i, ax in enumerate(axes.flat):
    abbrev, unit, colour = meta_vals[i]

    # Variable curve
    ax.plot(clim_mean[i], linewidth=1.5, color=colour,
            label=f'{abbrev}  –  {varnames[i]}')

    # Vertical event markers
    added_labels = set()
    for t, ev in enumerate(evlbl):
        ev = int(ev)
        if ev == 0:
            continue
        lbl = EVENT_LABEL_NAMES[ev] if ev not in added_labels else '_nolegend_'
        ax.axvline(t, color=EVENT_LABEL_COLORS[ev],
                   linestyle='--', alpha=0.7, linewidth=1.0, label=lbl)
        added_labels.add(ev)

    # Axes labels and title
    ax.set_title(f'[{i}]  {varnames[i]}  ({abbrev})', fontsize=9, fontweight='bold')
    ax.set_ylabel(unit, fontsize=8)
    ax.tick_params(labelsize=7)

    # Legend inside each subplot (variable name + any event markers present)
    ax.legend(fontsize=7, loc='upper left', framealpha=0.6)

# Shared x-axis label
for ax in axes[-1]:
    ax.set_xlabel('Timestep (daily ERA-5 steps)', fontsize=9)

# Global legend for event colors
from matplotlib.lines import Line2D
event_handles = [
    Line2D([0], [0], color=c, linestyle='--', linewidth=1.5, label=l)
    for ev, (l, c) in {k: (v, EVENT_LABEL_COLORS[k])
                        for k, v in EVENT_LABEL_NAMES.items() if k > 0}.items()
]
fig.legend(handles=event_handles, title='Event markers',
           loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02), framealpha=0.9)

plt.suptitle(
    'ERA-5 climate variables – spatial mean over the 20×20 grid\n'
    f'Sample 42  |  T = {T} daily timesteps  |  '
    f'damage class: {list(train_dataset_vis.damage_classes.keys())[int(s[6])]}',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('climate_series.png', dpi=150, bbox_inches='tight')
plt.show()

# Print a compact summary table
print(f'{"Idx":>3}  {"Abbrev":>6}  {"ERA-5 variable name":<42}  {"Unit"}')
print('─' * 72)
for i, name in enumerate(varnames):
    abbrev, unit, _ = ERA5_META[name]
    print(f'{i:>3}  {abbrev:>6}  {name:<42}  {unit}')

🔧 **Explore augmentation**

Enable `augment=True` when building the dataset and visualise the same patch
multiple times. Which augmentations are active? Can you add a new one
(e.g. elastic distortion, Mixup between two patches of the same class)?

🔧 **Class imbalance strategy**

The four classes are very unequal in size.  Investigate `class_weights` in
`configs/config_baseline.yaml` and compare uniform weights against
inverse-frequency weights for the focal loss.


## 2) The Baseline Model

The **BaselineMultimodalModel** is a compact architecture that processes
satellite images and climate data *without* any event or event-label information.
The temporal attention mechanism learns which climate timesteps matter by
itself, purely from the supervision signal.

```
x_pre  (B, 3, 64, 64) ──► CNNBackbone ──► f_pre  (B, 256)
x_post (B, 3, 64, 64) ──► CNNBackbone ──► f_post (B, 256)  (shared weights)

x_clim (B,10, T,20,20)
  └─► SpatialPerTimestep  (B*T, 10,20,20)  →  (B, T, 64)
  └─► SimpleTemporalEncoder               →  (B, T, 128)
  └─► SimpleAttention                     →  f_clim (B, 128)  +  att (B, T)

cat(f_pre, f_post, f_clim)  →  (B, 640)  →  MLP(640→256→128→4)
```


In [ ]:
from models import create_model

baseline = create_model(
    'BaselineMultimodalModel',
    num_classes=4, dropout_rate=0.2, vis_dim=256, climate_dim=128,
)
baseline.to(DEVICE)

total   = sum(p.numel() for p in baseline.parameters())
trainable = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
print(baseline)
print(f'\nTotal parameters    : {total:,}  ({total/1e6:.2f} M)')
print(f'Trainable parameters: {trainable:,}')


### 2.1)  Verify the Forward Pass

In [ ]:
# Synthetic batch – no real data needed
B, T = 4, 47
x_pre   = torch.randn(B, 3, 64, 64).to(DEVICE)
x_post  = torch.randn(B, 3, 64, 64).to(DEVICE)
x_clim  = torch.randn(B, 10, T, 20, 20).to(DEVICE)

with torch.no_grad():
    logits, att = baseline(x_pre, x_post, x_clim)

print(f'logits shape     : {logits.shape}   (B, num_classes)')
print(f'att_weights shape: {att.shape}   (B, T)')
print(f'softmax output   : {F.softmax(logits[0], dim=0).cpu().numpy().round(3)}')
print(f'attention sum    : {att[0].sum().item():.4f}  (should be ≈ 1.0)')


🔧 **Inspect each sub-module**

Call `baseline.visual_backbone`, `baseline.climate_spatial`, etc. and
print the output of each stage for the synthetic batch above.  Verify that
shapes match the diagram.

🔧 **Change climate_dim**

Rebuild the baseline with `climate_dim=64` and `climate_dim=256`.
How does this affect parameter count and throughput?


## 3) Training

We use **PyTorch Lightning** to keep boilerplate minimal.
The `DIA` Lightning module handles the training loop, metric logging to
**Weights & Biases**, and confusion-matrix visualisation.

The training command is one line in a terminal, or run the cell below.


In [ ]:
# ── 3.1  W&B login (run once) ────────────────────────────────────────────────
import wandb
# wandb.login()   # uncomment if you have not logged in yet


In [ ]:
# ── 3.2  Train from the command line (recommended) ───────────────────────────
# Open a terminal in the project root and run:
#
#   python main.py \
#       --arch BaselineMultimodalModel \
#       --config configs/config_baseline.yaml
#

In [ ]:
# ── 3.3  Or train programmatically from the notebook ─────────────────────────
import yaml, lightning as L
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.callbacks import (ModelCheckpoint, EarlyStopping,
                                          LearningRateMonitor)
from lightning_module import DIA

with open(os.path.join(PROJECT_DIR, 'configs/config_baseline.yaml')) as f:
    cfg = yaml.safe_load(f)

cfg['data']['cache_dir']   = CACHE_DIR
cfg['data']['num_workers'] = 2    # lower for notebook

dia_model = DIA(cfg, 'BaselineMultimodalModel', 'xBDClimate')

# Quick 2-epoch run to verify the pipeline (set to your desired value)
cfg['trainer']['epochs'] = 2

# ── Loggers ───────────────────────────────────────────────────────────────────
# CSVLogger writes loss and metrics to ./csv_logs/<version>/metrics.csv
# so that training curves can be plotted locally without W&B.
# Switch USE_WANDB to True (and run `wandb login` first) for full experiment
# tracking with hyperparameter sweeps and confusion-matrix images.
USE_WANDB = False

if USE_WANDB:
    logger = WandbLogger(project='DIA', name='baseline-notebook')
    extra_callbacks = [LearningRateMonitor(logging_interval='epoch')]
else:
    logger = CSVLogger(save_dir='.', name='csv_logs')
    extra_callbacks = []   # LearningRateMonitor needs a logger that supports it

print(f'Logger : {type(logger).__name__}')
print(f'Log dir: {logger.log_dir}')

# ── Callbacks ─────────────────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    dirpath='./checkpoints_nb',
    filename='best-{epoch}-{val_f1_macro:.3f}',
    monitor='val_f1_macro', mode='max', save_top_k=1,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = L.Trainer(
    max_epochs=cfg['trainer']['epochs'],
    accelerator=DEVICE,
    devices=1,
    callbacks=[checkpoint_cb] + extra_callbacks,
    enable_progress_bar=True,
    limit_train_batches=100,   # [!] limited data for this demo
    limit_val_batches=100,
    log_every_n_steps=10,
    logger=logger,              # CSVLogger or WandbLogger
)

trainer.fit(dia_model)
print(f'Training complete.  Metrics written to: {logger.log_dir}')


### Inspect Training Curves

In [ ]:
# ── 3.4  Inspect logged metrics after training ───────────────────────────────
# Lightning accumulates every value passed to self.log() in two dicts:
#   trainer.logged_metrics   – last value seen for each key
#   trainer.callback_metrics – same, also used by callbacks (checkpointing, etc.)
# Keys follow the pattern:  {split}_{metric}
#   e.g. train_loss, train_f1_macro, val_loss, val_f1_macro, val_f1_weighted

print('Metrics available after training:')
print('─' * 50)
for k, v in sorted(trainer.logged_metrics.items()):
    print(f'  {k:<25}: {float(v):.4f}')

# ── 3.5  Plot training curves from the CSV logger ────────────────────────────
# When logger=False, Lightning still writes a local CSV if CSVLogger is added.
# Here we reconstruct the curves directly from trainer.logged_metrics history
# stored in the EarlyStopping / Checkpoint callback, or from a CSVLogger file.
# For a full run use wandb.Api() or enable CSVLogger; below we show the pattern.

import pandas as pd, os, glob

# Look for a Lightning CSV log (produced by CSVLogger if used)
csv_files = sorted(glob.glob('./csv_logs/version_*/metrics.csv'))

if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'\nCSV log found: {csv_files[-1]}')
    print(df.tail())

    # Separate epoch-level rows (step is NaN for epoch metrics)
    epoch_df = df.dropna(subset=['epoch']).groupby('epoch').last().reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    metrics = [
        ('train_loss',       'val_loss',        'Loss',      '#e53935', '#1e88e5'),
        ('train_f1_macro',   'val_f1_macro',    'F1 Macro',  '#43a047', '#8e24aa'),
        ('train_f1_weighted','val_f1_weighted',  'F1 Weighted','#fb8c00','#00acc1'),
    ]

    for ax, (tr_key, va_key, title, c_tr, c_va) in zip(axes, metrics):
        if tr_key in epoch_df.columns:
            ax.plot(epoch_df['epoch'], epoch_df[tr_key],
                    label='Train', color=c_tr, linewidth=2)
        if va_key in epoch_df.columns:
            ax.plot(epoch_df['epoch'], epoch_df[va_key],
                    label='Val', color=c_va, linewidth=2, linestyle='--')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle('Training curves – BaselineMultimodalModel', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

else:
    # No CSV logger: show what we know from the final logged values
    print('\nNo CSV log found (logger=False was used).')
    print('For full training curves, either:')
    print('  a) Run via main.py  →  W&B dashboard (https://wandb.ai)')
    print('  b) Add CSVLogger to the Trainer:')
    print('       from lightning.pytorch.loggers import CSVLogger')
    print('       trainer = L.Trainer(..., logger=CSVLogger("."))')
    print()

    # Show a bar chart of the final metrics we do have
    metrics_dict = {k: float(v) for k, v in trainer.logged_metrics.items()
                    if any(m in k for m in ['loss','f1'])}
    if metrics_dict:
        fig, ax = plt.subplots(figsize=(8, 4))
        colours = ['#e53935' if 'loss' in k else
                   '#43a047' if 'train' in k else '#1e88e5'
                   for k in metrics_dict]
        bars = ax.barh(list(metrics_dict.keys()), list(metrics_dict.values()),
                       color=colours)
        ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
        ax.set_xlabel('Value')
        ax.set_title('Final logged metrics (end of training)', fontweight='bold')
        ax.set_xlim(0, max(metrics_dict.values()) * 1.15)
        plt.tight_layout()
        plt.savefig('final_metrics.png', dpi=150, bbox_inches='tight')
        plt.show()

# ── 3.6  Model summary ───────────────────────────────────────────────────────
from lightning.pytorch.utilities.model_summary import ModelSummary
print(ModelSummary(dia_model, max_depth=3))


🔧 **Learning rate schedule**

Plot the learning rate over 100 epochs for `CosineAnnealingLR`.
Then implement a warmup stage (first 5 epochs ramp from `lr/10` to `lr`)
and compare convergence speed.

🔧 **Focal loss vs cross-entropy**

Replace `FocalLoss` in `lightning_module.py` with plain `nn.CrossEntropyLoss`
with inverse-frequency class weights.  Does F1 improve on the minority class?

🔧 **Mixed-precision training**
Set `precision: '16-mixed'` in the config.  Verify that throughput increases
(should be ~2×) and that final F1 is within 0.01 of the 32-bit baseline.


## 4) Evaluation

We evaluate on the held-out **validation set** during training and on the
**test set** after training is finished.  The primary metric is
**F1 macro** (unweighted average over the four classes).

Key metrics:
| Metric | Formula | Notes |
|--------|---------|-------|
| F1 macro | mean(F1 per class) | Main competition metric |
| F1 weighted | weighted mean | Accounts for class frequency |
| Confusion matrix | recall per cell | Shows per-class errors |


In [ ]:
# ── 4.1  Load a checkpoint and evaluate on validation set ────────────────────
CKPT_PATH = './checkpoints_nb/best-*.ckpt'   # glob or explicit path
import glob
ckpts = sorted(glob.glob(CKPT_PATH))
if not ckpts:
    print('No checkpoint found – run Part 3 first.')
else:
    ckpt_path = ckpts[-1]
    print(f'Loading: {ckpt_path}')

    with open(os.path.join(PROJECT_DIR, 'configs/config_baseline.yaml')) as f:
        cfg = yaml.safe_load(f)
    cfg['data']['cache_dir']   = CACHE_DIR
    cfg['data']['num_workers'] = 2

    trained_model = DIA.load_from_checkpoint(
        ckpt_path, cfg=cfg, arch='BaselineMultimodalModel', database='xBDClimate'
    )
    trained_model.eval().to(DEVICE)
    print('Checkpoint loaded.')


In [ ]:
# ── 4.2  Run inference on validation set ────────────────────────────────────
from databases.xBDClimate_database import get_dataloaders
from torch.utils.data import DataLoader

_, val_loader, _ = get_dataloaders(
    cache_dir=CACHE_DIR, batch_size=64, num_workers=2, augment=False
)

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in val_loader:
        pre, post, mask, clim, ev, lbl_pre, lbl_post, _, _ = batch
        pre, post, clim, ev = pre.to(DEVICE), post.to(DEVICE), clim.to(DEVICE), ev.to(DEVICE)

        model_out = trained_model.model(pre, post, clim, ev)
        logits    = model_out[0] if isinstance(model_out, tuple) else model_out

        all_probs .append(F.softmax(logits, dim=1).cpu())
        all_preds .append(logits.argmax(dim=1).cpu())
        all_labels.append(lbl_post.cpu())

        if len(all_probs) > 100:  # [!] limit for demo; remove for full evaluation
            break

all_preds  = torch.cat(all_preds ).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = torch.cat(all_probs ).numpy()

damage_names = list(val_loader.dataset.damage_classes.keys())

f1_macro    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
f1_per_cls  = f1_score(all_labels, all_preds, average=None,       zero_division=0)

print(f'F1 macro    : {f1_macro:.4f}')
print(f'F1 weighted : {f1_weighted:.4f}')
print('F1 per class:')
for n, f in zip(damage_names, f1_per_cls):
    print(f'  {n:<15}: {f:.4f}')


### Confusion Matrix

In [ ]:
cm     = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, fmt in zip(
        axes,
        [cm, cm_pct],
        ['Confusion matrix (counts)', 'Confusion matrix (row-normalised)'],
        ['d', '.2f']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=damage_names, yticklabels=damage_names, ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


### Per-Class Precision, Recall, F1

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds,
                             target_names=damage_names, zero_division=0))


🔧 **Error analysis by disaster**

Group the mis-classified samples by disaster event name
(available in the batch as `img_path`).
Which disasters are hardest?  Is it related to the disaster type,
the satellite resolution, or the climate representation?


## 5) Interpretation and Explainability

We use two complementary tools:

1. **Attention weights** – the model's own soft focus over climate timesteps.
2. **Integrated Gradients (IG)** – gradient-based attribution that assigns
   importance scores to every input pixel and climate variable.

Both are already wired into the system via `lightning_module.interpretation()`.


### 5.1) Run the Full Interpretation Pipeline

In [ ]:
# The easiest way: point config_interpretation.yaml at your checkpoint and run
#
#   python main.py \
#       --arch BaselineMultimodalModel \
#       --config configs/config_interpretation.yaml
#
# This produces two PNG files per correctly-predicted sample:
#   {disaster}_p{idx}_satellite.png   – pre/post images + RGB attributions
#   {disaster}_p{idx}_climate.png     – climate curves + IG + attention
#
print('See terminal command above.')
print('Interpretation plots saved to experiments/.../  directory.')


### 5.2)  Visualize Attention Weights Interactively

In [ ]:
from databases.xBDClimate_database import xBDClimateDataset, FilteredSubset

# Load training split filtered to 'destroyed' (class 3)
train_raw = xBDClimateDataset(os.path.join(CACHE_DIR,'cache_train'), augment=False)
destroyed = FilteredSubset(train_raw, target_cat=3)
print(f'Destroyed patches: {len(destroyed)}')

loader_det = torch.utils.data.DataLoader(destroyed, batch_size=8, shuffle=False)
batch      = next(iter(loader_det))
pre, post, mask, clim, ev, lbl_pre, lbl_post, img_paths, patch_idxs = batch
pre, post, clim, ev = pre.to(DEVICE), post.to(DEVICE), clim.to(DEVICE), ev.to(DEVICE)

with torch.no_grad():
    logits, att_weights = trained_model.model(pre, post, clim, ev)

probs = F.softmax(logits, dim=1).cpu().numpy()
att   = att_weights.cpu().numpy()          # (B, T)
T     = att.shape[1]
varnames = train_raw.climate_variable_names

fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(att): break
    ax.fill_between(range(T), att[i], alpha=0.6, color='#2196F3')
    ax.set_ylim(0, att[i].max()*1.4 if att[i].max()>0 else 0.1)
    ax.set_title(
        f'pred={probs[i].argmax()}  gt={int(lbl_post[i])}\n'
        f'p(destroyed)={probs[i,3]:.2f}', fontsize=8)
    ax.set_xlabel('Timestep', fontsize=8)
    ax.set_ylabel('Attention', fontsize=8)

plt.suptitle('Self-attention weights over climate timesteps (destroyed class)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_weights_destroyed.png', dpi=150)
plt.show()


### 5.3)  Integrated Gradients on Satellite Images

In [ ]:
from captum.attr import IntegratedGradients
from lightning_module import CaptumWrapper

wrapped = CaptumWrapper(trained_model.model, output_idx=0)
ig      = IntegratedGradients(wrapped)

# Take the first sample from the batch
s = 0
inp_pre  = pre[[s]].requires_grad_(True)
inp_post = post[[s]].requires_grad_(True)
inp_clim = clim[[s]].requires_grad_(True)
ev_s     = ev[[s]]

pred_cls = int(logits[s].argmax().cpu())
print(f'Predicted class: {pred_cls} ({damage_names[pred_cls]})')
print(f'True class     : {int(lbl_post[s])} ({damage_names[int(lbl_post[s])]})')

attr_pre, attr_post, attr_clim = ig.attribute(
    inputs=(inp_pre, inp_post, inp_clim),
    baselines=(torch.zeros_like(inp_pre),
               torch.zeros_like(inp_post),
               torch.zeros_like(inp_clim)),
    additional_forward_args=(ev_s,),
    target=pred_cls,
    n_steps=50,
    internal_batch_size=1,
)

# Aggregate RGB attributions: absolute sum over channels
attr_pre_vis  = attr_pre[0].abs().sum(0).cpu().detach().numpy()
attr_post_vis = attr_post[0].abs().sum(0).cpu().detach().numpy()

# Satellite images (denormed)
mean_np = trained_model.data_train.dataset.patch_mean if hasattr(trained_model.data_train, 'dataset') else train_raw.patch_mean
std_np  = trained_model.data_train.dataset.patch_std  if hasattr(trained_model.data_train, 'dataset') else train_raw.patch_std
img_pre  = denorm(pre[s].cpu(), mean_np, std_np)
img_post = denorm(post[s].cpu(), mean_np, std_np)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

axes[0,0].imshow(img_pre);                axes[0,0].set_title('Pre-disaster image')
im = axes[0,1].imshow(attr_pre_vis,  cmap='hot'); axes[0,1].set_title('IG attribution – pre')
fig.colorbar(im, ax=axes[0,1])
axes[0,2].imshow(img_pre); axes[0,2].imshow(attr_pre_vis,  alpha=0.6, cmap='hot')
axes[0,2].set_title('Overlay – pre')

axes[1,0].imshow(img_post);               axes[1,0].set_title('Post-disaster image')
im = axes[1,1].imshow(attr_post_vis, cmap='hot'); axes[1,1].set_title('IG attribution – post')
fig.colorbar(im, ax=axes[1,1])
axes[1,2].imshow(img_post); axes[1,2].imshow(attr_post_vis, alpha=0.6, cmap='hot')
axes[1,2].set_title('Overlay – post')

for ax in axes.flat: ax.axis('off')
plt.suptitle(
    f'Integrated Gradients – pred: {damage_names[pred_cls]}, '
    f'true: {damage_names[int(lbl_post[s])]}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ig_satellite.png', dpi=150)
plt.show()


### 5.4)  Integrated Gradients on Climate Variables

In [ ]:
# Spatially-average climate IG attributions: (1,10,T,20,20) → (T,10)
attr_clim_mean = attr_clim[0].abs().mean(dim=(2,3)).cpu().detach().numpy()  # (10,T)

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.fill_between(range(T), attr_clim_mean[i], alpha=0.7, color='#FF5722')
    ax2 = ax.twinx()
    clim_curve = clim[s, i].mean(dim=(1,2)).cpu().numpy()   # spatial mean
    ax2.plot(clim_curve, linewidth=1, color='#2196F3', alpha=0.6)
    ax.set_title(varnames[i], fontsize=9)
    ax.set_ylabel('IG attribution', fontsize=8)
    ax2.set_ylabel('Input value', fontsize=8, color='#2196F3')

plt.suptitle('IG attribution (red) vs input value (blue) per climate variable',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ig_climate.png', dpi=150)
plt.show()


🔧 **GradCAM on satellite images**  
Implement GradCAM (or ScoreCAM) using the final conv layer of the
`CNNBackbone`.  Overlay the heatmap on the post-disaster image.
Does the model look at the building or its surroundings?

🔧 **Class-specific attention**
Repeat the attention analysis from §5.2 for *all four* damage classes.
Is the attention distribution different for `no-damage` vs `destroyed`?
What climate variable is most attended?

🔧 **Saliency for failure cases**  
Select 4–6 samples where the model is confidently wrong.
Apply IG and visualise the attributions.  Can you hypothesise why the
model fails?


## 6) Research Directions and Challenge Ideas

This section is your **starting point for novel contributions**.

---


### 6.A) Architecture: Visual Branch

- Siamese difference stream: Instead of concatenating f_pre and f_post, feed their element-wise difference and product as additional streams.
- Replace CNNBackbone with a frozen ResNet-18 or EfficientNet-B0, fine-tune only the final layers. Does ImageNet pre-training help for aerial/satellite imagery?
- (Tiny) Vision Transformer (ViT-Tiny). Compare attention maps from ViT vs Integrated Gradients.
- Contrastive pre-training and fine-tuning on the damage labels.

### 6.B) Architecture: Climate Branch

- Transformer encoder
- Mamba state-space model 
- Seasonal decomposition: Decompose the climate series into trend + seasonal + residual components and feed them as separate channels. Does the residual (anomaly) alone give a better signal?
- Spatial convolution over the 20×20 ERA-5 grid: Use a small U-Net on the 20×20 spatial grid to encode regional climate context instead of a single global average.

### 6.C) Architecture: Fusion
- Cross-modal attention
- Gated fusio: Learn a gate g = sigmoid(W_g [f_vis; f_clim]) and compute output = g * f_vis + (1-g) * f_clim. Does the gate learn to rely more on climate for certain disaster types?
- Late fusion with uncertainty: Train separate visual and climate classifiers, then combine their softmax outputs with learned uncertainty weights (E.g., using Monte Carlo-Dropout).

### 6.D) Training Strategy

- Mixup for multimodal input: Apply Mixup independently in each modality (satellite, climate) and train with soft labels.
- Curriculum learning: Sort training samples by difficulty (entropy of teacher predictions). Start with easy examples and gradually include hard ones. Compare with random shuffling.
- Knowledge distillation: Train a compact student (baseline) to mimic the soft logits of a larger teacher.
- Class-balanced sampling: Implement a WeightedRandomSampler that over-samples minority classes. Compare against focal-loss reweighting.

### 6.E) Interpretability Deep-Dives
- SHAP for climate variables: Which ERA-5 variable has the highest Shapley value per damage class?
- Define 5–8 interpretable concepts (e.g. 'roof intact','flooding visible','wind damage') as intermediate targets. Train a concept layer between the backbone and the damage classifier.
- Counterfactual generation: For a 'destroyed' building, find the minimal perturbation of the post-image that changes the prediction to 'no-damage'.

### 6.F) Novel Applications
- Disaster-type generalization: Split data, e.g., train on hurricanes + earthquakes, test on wildfires + floods.
- Temporal forecasting: Instead of classifying damage from the post-image, predict damage before the post-image is available, using only the pre-image + forecast climate.  This is a truly prospective risk model.
- Uncertainty quantification: E.g., Monte-Carlo Dropout. Plot the epistemic uncertainty map on the satellite image. Is uncertainty higher in areas with ambiguous damage?

---

## 🏆 Challenge Presentation

### What to hand in

| File | Description |
|------|-------------|
| `models/my_model.py` | Your improved model class |
| `configs/my_config.yaml` | Hyperparameters used |
| `best.ckpt` | Best checkpoint |
| W&B run link | Experiment tracking |
| Presentation (5 slides) | What you tried and what you learned |
---

**Good luck!  Build something you're proud of.** 🚀